# THE GOAT DEBATE — NBA GOAT Calculator & Career Trend Analysis
**Michael Jordan · Kobe Bryant · LeBron James**

An interactive, dark-mode dashboard built from the processed game-log summaries in this repo
(`data/processed/game_logs_summary_career_overall.csv` and `game_logs_summary_per_season.csv`).

| Part | What it does |
|---|---|
| **1 · GOAT Calculator** | Weight 13 categories (stats + accolades) on a 1–10 scale, fine-tune every individual stat, and watch the GOAT score update live, contribution breakdown, trophy case, radar, sensitivity curve and a 10,000-scenario robustness check. Career-volume totals (total points, total games, seasons played) are deliberately **not** scored. |
| **2 · Career Trend Analysis** | 9 interactive views of how each career evolved, plus 9 auto-generated insight cards computed straight from the data. |

**How to run:** `Run All` (Kernel ▸ Restart & Run All). The notebook looks for the CSVs in `../data/processed`, `data/processed`, `data`, then the current folder.
Interactivity uses `ipywidgets`, so run it in JupyterLab / Jupyter Notebook 7 / VS Code (not a static viewer). Static PNG snapshots are written to `notebooks/assets/` at the very end for GitHub previews.

In [ ]:
# ── 0 · CONFIG, PALETTE & STYLE ───────────────────────────────────────────────
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as W
from IPython.display import display, HTML, Image, clear_output

warnings.filterwarnings("ignore", category=FutureWarning)

# --- where the data lives (first folder containing BOTH files wins) -------------
DATA_DIR_CANDIDATES = [Path("../data/processed"), Path("data/processed"), Path("data"), Path(".")]
CAREER_FILE = "game_logs_summary_career_overall.csv"
SEASON_FILE = "game_logs_summary_per_season.csv"

# --- behaviour switches ----------------------------------------------------------
PLOTLY_RENDERER = "plotly_mimetype+notebook_connected"   # use "notebook" for fully offline use
EXPORT_STATIC = True                                      # PNG snapshots (needs: pip install kaleido + Chrome)
ASSETS_DIR = Path("assets")
MIN_GAMES_QUALIFIED = 30                                  # seasons below this are flagged "low-game"
MC_SIMULATIONS = 10_000
MC_SEED = 42
WEIGHT_MIN, WEIGHT_MAX, WEIGHT_DEFAULT = 1, 10, 5   # slider scale (set WEIGHT_MIN = 0 if you want 0 to mean "switch this off")
HONORS_AS_RATES = True      # True: All-Star/All-NBA/All-Defensive scored per season played (no longevity bonus); False: raw counts

pio.renderers.default = PLOTLY_RENDERER

# Live in-place chart updates need `anywidget` (pip install anywidget). Without it the charts simply redraw on every change.
try:
    import anywidget  # noqa: F401
    go.FigureWidget(go.Figure())
    LIVE_PLOTS = True
except Exception:
    LIVE_PLOTS = False
    print("anywidget not found: charts will redraw on every slider change (slower). For smooth live updates run: pip install anywidget")

# --- palette ---------------------------------------------------------------------
THEME = dict(
    bg="#0B0F1A", panel="#121A2B", panel_alt="#17223A", border="#1F2A44",
    text="#E8ECF4", muted="#8A96B0", grid="#1B2540",
    nba_blue="#1D428A", nba_blue_light="#3B6FD4", nba_red="#C8102E",
)
PLAYERS = ["Michael Jordan", "Kobe Bryant", "LeBron James"]
PLAYER_COLORS = {"Michael Jordan": "#CE1141",   # Bulls red
                 "Kobe Bryant":    "#8A5CC9",   # Lakers purple (lightened for dark UI)
                 "LeBron James":   "#FDBB30"}   # Cavaliers gold
PLAYER_SLUG = {"Michael Jordan": "mj", "Kobe Bryant": "kb", "LeBron James": "lj"}
SYMBOLS = {"Michael Jordan": "circle", "Kobe Bryant": "diamond", "LeBron James": "square"}
DASHES = {"Michael Jordan": "solid", "Kobe Bryant": "dash", "LeBron James": "dashdot"}
NAME_FIX = {"Lebron James": "LeBron James"}
FONT = "Inter, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"
PLOT_CONFIG = dict(displaylogo=False, modeBarButtonsToRemove=["lasso2d", "select2d"])


def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


def style(fig, title=None, height=430, legend=True, margin=None):
    """Apply the dark NBA theme to a Plotly figure."""
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=THEME["panel"], plot_bgcolor=THEME["panel"],
        font=dict(family=FONT, color=THEME["text"], size=12),
        title=dict(text=title, x=0.01, xanchor="left", font=dict(size=16)) if title else None,
        height=height,
        margin=margin or dict(l=60, r=30, t=70 if title else 40, b=55),
        showlegend=legend,
        legend=dict(orientation="h", x=1, xanchor="right", y=1.0, yanchor="bottom", bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(bgcolor=THEME["bg"], bordercolor=THEME["border"], font=dict(color=THEME["text"])),
    )
    fig.update_xaxes(gridcolor=THEME["grid"], zerolinecolor=THEME["grid"], linecolor=THEME["border"])
    fig.update_yaxes(gridcolor=THEME["grid"], zerolinecolor=THEME["grid"], linecolor=THEME["border"])
    return fig


def show(fig):
    fig.show(config=PLOT_CONFIG)


def caption(text):
    return HTML(f'<div class="cap">{text}</div>')


# --- CSS: dashboard cards + dark skin for the ipywidgets controls ------------------
CSS = """
<style>
.goat-banner{background:linear-gradient(135deg,@panel 0%,@bg 100%);border:1px solid @border;border-radius:12px;
  padding:22px 26px 0;margin:6px 0 14px;overflow:hidden;font-family:@font}
.gb-title{font-size:30px;font-weight:800;letter-spacing:.06em;color:@text}
.gb-title span{color:@nba_red}
.gb-sub{color:@muted;font-size:14px;margin:4px 0 16px}
.gb-stripe{height:6px;margin:0 -26px;background:linear-gradient(90deg,@nba_red 0 33%,@text 33% 34%,@nba_blue 34% 100%)}
.pc-grid,.podium{display:grid;grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:14px;margin:8px 0;font-family:@font}
.pc,.pod{background:@panel;border:1px solid @border;border-top:5px solid var(--c);border-radius:10px;padding:14px 16px;color:@text}
.pc h3{margin:0 0 2px;font-size:18px;color:var(--c)}
.pc .sub{color:@muted;font-size:12px;margin-bottom:10px}
.pc .stats{display:grid;grid-template-columns:repeat(4,1fr);gap:6px;text-align:center}
.pc .stats b{display:block;font-size:20px}
.pc .stats span{font-size:10.5px;color:@muted;text-transform:uppercase;letter-spacing:.05em}
.pod{text-align:center}
.pod.first{box-shadow:0 0 0 1px var(--c),0 8px 28px rgba(0,0,0,.45)}
.pod .rank{font-size:12px;font-weight:700;letter-spacing:.14em;color:@muted}
.pod .score{font-size:44px;font-weight:800;color:var(--c);line-height:1.1}
.pod .name{font-size:17px;font-weight:700}
.pod .gap{font-size:12px;color:@muted}
.verdict{background:@panel_alt;border-left:4px solid @nba_red;border-radius:6px;padding:12px 16px;margin:6px 0 12px;
  color:@text;font-size:15px;font-family:@font}
.warnbox{background:#3a1018;border-left:4px solid @nba_red;border-radius:6px;padding:8px 14px;margin:6px 0;color:@text;font-size:13px}
.cap{color:@muted;font-size:12.5px;margin:2px 4px 14px;font-family:@font}
.sec{color:@text;font-size:13px;letter-spacing:.12em;text-transform:uppercase;font-weight:700;margin:16px 0 4px;
  border-bottom:2px solid @nba_blue;padding-bottom:4px;font-family:@font}
.ins-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:14px;font-family:@font}
.ins{background:@panel;border:1px solid @border;border-left:4px solid @nba_blue_light;border-radius:10px;padding:14px 16px;color:@text}
.ins .t{font-size:11px;letter-spacing:.1em;color:@muted;text-transform:uppercase}
.ins .h{font-size:16px;font-weight:700;margin:3px 0 8px}
.ins .l{font-size:13px;margin:5px 0;line-height:1.45}
.bp{background:@panel;border:1px solid @border;border-radius:10px;padding:14px 22px;color:@text;font-family:@font;font-size:14px;line-height:1.6}
.methtbl{border-collapse:collapse;color:@text;font-size:13px;font-family:@font}
.methtbl td,.methtbl th{border-bottom:1px solid @border;padding:5px 12px;text-align:left}
.goat-root{background:@bg;border:1px solid @border;border-radius:12px;padding:14px}
.goat-root .widget-label,.goat-root .widget-readout,.goat-root label{color:@text !important}
.goat-root select,.goat-root input[type=text]{background:@panel;color:@text;border:1px solid @border}
.goat-root button.jupyter-button{background:@nba_blue;color:#fff;border:none}
.goat-root button.jupyter-button:hover{background:@nba_red}
.goat-root button.jupyter-button.mod-active{background:@nba_red}
.goat-root input[type=range]{accent-color:@nba_red}
.goat-root .widget-checkbox input{accent-color:@nba_red}
__PLAYER_CSS__
</style>
"""
_player_css = "\n".join(f".pbox-{PLAYER_SLUG[p]} label{{color:{c} !important;font-weight:700}}" for p, c in PLAYER_COLORS.items())
CSS = CSS.replace("__PLAYER_CSS__", _player_css).replace("@font", FONT)
for _k in sorted(THEME, key=len, reverse=True):
    CSS = CSS.replace("@" + _k, THEME[_k])
display(HTML(CSS))


def banner_html(title_html, subtitle):
    return (f'<div class="goat-banner"><div class="gb-title">{title_html}</div>'
            f'<div class="gb-sub">{subtitle}</div><div class="gb-stripe"></div></div>')


display(HTML(banner_html('THE <span>GOAT</span> DEBATE',
                         'Michael Jordan vs Kobe Bryant vs LeBron James · career &amp; season game-log analysis')))

## 1 · Load, validate & engineer features
Key facts about this data (all handled below):
* Shooting totals and per-game averages **combine regular season + playoffs**; games and wins are split by season type, wins are regular-season only.
* The per-season file has **no points column** — it is derived as `2·FGM + 3PM + FTM` and checked against the career `PTS_AVE`.
* No championships, MVPs, All-Star selections, advanced metrics (PER/WS/BPM) or ages exist in the data, so none are used or invented.

In [ ]:
def find_data_dir():
    for d in DATA_DIR_CANDIDATES:
        if (d / CAREER_FILE).exists() and (d / SEASON_FILE).exists():
            return d
    raise FileNotFoundError(
        f"Could not find {CAREER_FILE} and {SEASON_FILE} in any of: "
        + ", ".join(str(p) for p in DATA_DIR_CANDIDATES)
        + ". Edit DATA_DIR_CANDIDATES in the config cell.")


DATA_DIR = find_data_dir()
CAREER = pd.read_csv(DATA_DIR / CAREER_FILE)
SEASON = pd.read_csv(DATA_DIR / SEASON_FILE)
for _df in (CAREER, SEASON):
    _df["PLAYER_NAME"] = _df["PLAYER_NAME"].str.strip().replace(NAME_FIX)


def add_season_features(df):
    d = df.copy()
    gp = d["GAMES_PLAYED_TOTAL"]
    d["SEASON_START"] = d["SEASON_YEAR"].str[:4].astype(int)
    d["PTS_TOTAL"] = 2 * d["TOTAL_FGM"] + d["TOTAL_FG3M"] + d["TOTAL_FTM"]
    d["PPG"] = d["PTS_TOTAL"] / gp
    d["EFG"] = (d["TOTAL_FGM"] + 0.5 * d["TOTAL_FG3M"]) / d["TOTAL_FGA"]
    d["TS"] = d["PTS_TOTAL"] / (2 * (d["TOTAL_FGA"] + 0.44 * d["TOTAL_FTA"]))
    d["FG3A_PG"] = d["TOTAL_FG3A"] / gp
    d["FTA_PG"] = d["TOTAL_FTA"] / gp
    d["FG3A_RATE"] = d["TOTAL_FG3A"] / d["TOTAL_FGA"]
    d["FT_RATE"] = d["TOTAL_FTA"] / d["TOTAL_FGA"]
    d["PLAYOFF_SHARE"] = d["GAMES_PLAYED_PLAYOFFS"] / gp
    d["STOCKS"] = d["STL_AVE"] + d["BLK_AVE"]
    d["AST_TOV"] = d["AST_AVE"] / d["TOV_AVE"]
    d["QUALIFIED"] = gp >= MIN_GAMES_QUALIFIED
    d["LOW_GAMES"] = ~d["QUALIFIED"]
    return d.sort_values(["PLAYER_NAME", "PLAYER_SEASON_NUM"]).reset_index(drop=True)


def add_career_features(df):
    d = df.copy()
    d["PTS_TOTAL"] = 2 * d["TOTAL_FGM"] + d["TOTAL_FG3M"] + d["TOTAL_FTM"]
    d["EFG"] = (d["TOTAL_FGM"] + 0.5 * d["TOTAL_FG3M"]) / d["TOTAL_FGA"]
    d["TS"] = d["PTS_TOTAL"] / (2 * (d["TOTAL_FGA"] + 0.44 * d["TOTAL_FTA"]))
    d["AST_TOV"] = d["AST_AVE"] / d["TOV_AVE"]
    d["STOCKS"] = d["STL_AVE"] + d["BLK_AVE"]
    d["GP_PER_SEASON"] = d["GAMES_PLAYED_REGULAR_SEASON"] / d["TOTAL_SEASONS_PLAYED"]
    d["PO_SHARE"] = d["GAMES_PLAYED_PLAYOFFS"] / d["GAMES_PLAYED_TOTAL"]
    d["PO_SEASON_RATE"] = d["TOTAL_PLAYOFF_SEASONS_PLAYED"] / d["TOTAL_SEASONS_PLAYED"]
    d["FOULED_OUT_RATE"] = d["FOULED_OUT_TOTAL"] / d["GAMES_PLAYED_TOTAL"] * 100   # per 100 games
    return d


SEASON = add_season_features(SEASON)
CAREER = add_career_features(CAREER)
SEASON_BY_PLAYER = {p: g.sort_values("PLAYER_SEASON_NUM").reset_index(drop=True) for p, g in SEASON.groupby("PLAYER_NAME")}


def peak_features(p):
    q = SEASON_BY_PLAYER[p].query("QUALIFIED")
    return dict(PEAK_PPG=q["PPG"].max(), PEAK_TS=q["TS"].max(),
                BEST3_PPG=q["PPG"].rolling(3).mean().max(),
                PPG_CV=q["PPG"].std(ddof=0) / q["PPG"].mean(), TS_STD=q["TS"].std(ddof=0))


CAREER_X = CAREER.set_index("PLAYER_NAME").loc[PLAYERS]
CAREER_X = CAREER_X.join(pd.DataFrame({p: peak_features(p) for p in PLAYERS}).T)

# --- validation ------------------------------------------------------------------
_agg = SEASON.groupby("PLAYER_NAME").agg(pts=("PTS_TOTAL", "sum"), gp=("GAMES_PLAYED_TOTAL", "sum"),
                                         fga=("TOTAL_FGA", "sum"), seasons=("SEASON_YEAR", "count"))
for p in PLAYERS:
    c = CAREER_X.loc[p]
    assert abs(_agg.loc[p, "pts"] / _agg.loc[p, "gp"] - c["PTS_AVE"]) < 0.01, f"PPG mismatch for {p}"
    assert _agg.loc[p, "gp"] == c["GAMES_PLAYED_TOTAL"], f"Games mismatch for {p}"
    assert _agg.loc[p, "fga"] == c["TOTAL_FGA"], f"FGA mismatch for {p}"
    assert _agg.loc[p, "seasons"] == c["TOTAL_SEASONS_PLAYED"], f"Season count mismatch for {p}"
assert set(PLAYERS) == set(SEASON["PLAYER_NAME"]) == set(CAREER["PLAYER_NAME"]), "Unexpected player names"


def calendar_gaps(p):
    d = SEASON_BY_PLAYER[p]
    miss = []
    for a, b in zip(d["SEASON_START"][:-1], d["SEASON_START"][1:]):
        miss += [f"{y}-{str(y + 1)[2:]}" for y in range(a + 1, b)]
    return ", ".join(miss) if miss else "none"


_cov = pd.DataFrame({
    "Seasons": [int(_agg.loc[p, "seasons"]) for p in PLAYERS],
    "First": [SEASON_BY_PLAYER[p]["SEASON_YEAR"].iloc[0] for p in PLAYERS],
    "Last": [SEASON_BY_PLAYER[p]["SEASON_YEAR"].iloc[-1] for p in PLAYERS],
    "Games (RS+PO)": [int(_agg.loc[p, "gp"]) for p in PLAYERS],
    "Low-game seasons (<%d GP)" % MIN_GAMES_QUALIFIED: [int(SEASON_BY_PLAYER[p]["LOW_GAMES"].sum()) for p in PLAYERS],
    "Calendar seasons not in data": [calendar_gaps(p) for p in PLAYERS],
}, index=PLAYERS)
print(f"Data folder: {DATA_DIR.resolve()}")
print("Validation passed: derived PPG, games, FGA and season counts reconcile between the two files.\n")
display(_cov)
SEASON.head(3)

## 1b · Accolades (external, editable)
Championships, awards and honours are **not** in the two CSVs, so they live in a small hand-compiled table below (nothing is scraped at run time).
* **As of 5 Nov 2026**, compiled from NBA.com player bios and year-by-year All-NBA pages, ESPN, and Wikipedia/Basketball-Reference career-achievement lists; cross-checked across at least two sources per number.
* LeBron's All-NBA streak ended at 21 in 2026: he played 60 games, short of the league's 65-game minimum. His 22nd All-Star selection (2026) is included.
* To correct or extend a number, edit the dict below **or** drop a CSV named `player_accolades.csv` in `data/external/` (same columns); the CSV overrides the embedded table.
* Games-based rates use the career file's `TOTAL_SEASONS_PLAYED`, so selections can be scored **per season played** instead of as raw totals (`HONORS_AS_RATES` in the config cell).

In [ ]:
ACC_FILE = "player_accolades.csv"
ACC_DIR_CANDIDATES = [Path("../data/external"), Path("data/external"), Path("data"), Path(".")]
ACC_AS_OF = "2026-11-05"

_ACC_EMBEDDED = {
    "Michael Jordan": dict(championships=6, finals_appearances=6, finals_mvp=6, mvp=5, dpoy=1, scoring_titles=10, allstar_mvp=3,
                           allstar_selections=14, allnba_first=10, allnba_second=1, allnba_third=0, alldefensive=9, olympic_gold=2,
                           champion_seasons="1990-91;1991-92;1992-93;1995-96;1996-97;1997-98",
                           mvp_seasons="1987-88;1990-91;1991-92;1995-96;1997-98"),
    "Kobe Bryant":    dict(championships=5, finals_appearances=7, finals_mvp=2, mvp=1, dpoy=0, scoring_titles=2, allstar_mvp=4,
                           allstar_selections=18, allnba_first=11, allnba_second=2, allnba_third=2, alldefensive=12, olympic_gold=2,
                           champion_seasons="1999-00;2000-01;2001-02;2008-09;2009-10",
                           mvp_seasons="2007-08"),
    "LeBron James":   dict(championships=4, finals_appearances=10, finals_mvp=4, mvp=4, dpoy=0, scoring_titles=1, allstar_mvp=3,
                           allstar_selections=22, allnba_first=13, allnba_second=4, allnba_third=4, alldefensive=6, olympic_gold=3,
                           champion_seasons="2011-12;2012-13;2015-16;2019-20",
                           mvp_seasons="2008-09;2009-10;2011-12;2012-13"),
}
ACC_NUM_COLS = ["championships", "finals_appearances", "finals_mvp", "mvp", "dpoy", "scoring_titles", "allstar_mvp",
                "allstar_selections", "allnba_first", "allnba_second", "allnba_third", "alldefensive", "olympic_gold"]


def load_accolades():
    df, src = pd.DataFrame(_ACC_EMBEDDED).T, "table embedded in this notebook"
    for d in ACC_DIR_CANDIDATES:
        f = d / ACC_FILE
        if f.exists():
            df = pd.read_csv(f)
            df["PLAYER_NAME"] = df["PLAYER_NAME"].str.strip().replace(NAME_FIX)
            df, src = df.set_index("PLAYER_NAME"), str(f.resolve())
            break
    missing = [c for c in ACC_NUM_COLS + ["champion_seasons", "mvp_seasons"] if c not in df.columns]
    assert not missing, f"Accolades table is missing columns: {missing}"
    df = df.loc[PLAYERS].copy()
    df[ACC_NUM_COLS] = df[ACC_NUM_COLS].astype(int)
    return df, src


ACC, ACC_SOURCE = load_accolades()
ACC["allnba_total"] = ACC["allnba_first"] + ACC["allnba_second"] + ACC["allnba_third"]
ACC["finals_win_pct"] = ACC["championships"] / ACC["finals_appearances"]
_seasons = CAREER_X["TOTAL_SEASONS_PLAYED"]
ACC["allstar_per_season"] = ACC["allstar_selections"] / _seasons
ACC["allnba_per_season"] = ACC["allnba_total"] / _seasons
ACC["allnba_first_per_season"] = ACC["allnba_first"] / _seasons
ACC["alldef_per_season"] = ACC["alldefensive"] / _seasons


def season_list(p, col):
    return [x for x in str(ACC.loc[p, col]).split(";") if x and x != "nan"]


# integrity checks on the external table
for p in PLAYERS:
    a = ACC.loc[p]
    assert a.championships <= a.finals_appearances, f"{p}: titles exceed Finals appearances"
    assert len(season_list(p, "champion_seasons")) == a.championships, f"{p}: champion_seasons length != championships"
    assert len(season_list(p, "mvp_seasons")) == a.mvp, f"{p}: mvp_seasons length != mvp"
    known = set(SEASON_BY_PLAYER[p]["SEASON_YEAR"])
    assert set(season_list(p, "champion_seasons") + season_list(p, "mvp_seasons")) <= known, f"{p}: award season not in season data"
    assert a.allnba_total >= a.allnba_first and a.allstar_selections <= _seasons[p], f"{p}: honours exceed seasons played"

CAREER_X = CAREER_X.join(ACC.drop(columns=["champion_seasons", "mvp_seasons"]))
print(f"Accolades source: {ACC_SOURCE} (as of {ACC_AS_OF}); integrity checks passed.")
display(ACC.drop(columns=["champion_seasons", "mvp_seasons"])[ACC_NUM_COLS + ["allnba_total"]].T.rename(index={
    "allnba_total": "allnba_total (derived)"}))

## 2 · Dashboard building blocks
Player cards, the "gap-aware" line helper (calendar gaps such as Jordan's retirements break the line instead of being interpolated) and shared chart utilities.

In [ ]:
def player_cards_html():
    cards = []
    for p in PLAYERS:
        c = CAREER_X.loc[p]
        stats = [(f"{c.PTS_AVE:.1f}", "PPG"), (f"{c.REB_AVE:.1f}", "RPG"), (f"{c.AST_AVE:.1f}", "APG"), (f"{c.TS:.1%}", "TS%")]
        st = "".join(f"<div><b>{v}</b><span>{k}</span></div>" for v, k in stats)
        cards.append(
            f'<div class="pc" style="--c:{PLAYER_COLORS[p]}"><h3>{p}</h3>'
            f'<div class="sub">{int(c.TOTAL_SEASONS_PLAYED)} seasons · {int(c.GAMES_PLAYED_TOTAL):,} games · '
            f'{c.REGULAR_SEASON_WIN_PCT:.1%} reg-season win rate</div>'
            f'<div class="sub" style="margin-top:-6px"><b>{int(c.championships)}</b> titles · <b>{int(c.mvp)}</b> MVP · '
            f'<b>{int(c.finals_mvp)}</b> Finals MVP · Finals {int(c.championships)}-{int(c.finals_appearances - c.championships)}</div>'
            f'<div class="stats">{st}</div></div>')
    return '<div class="pc-grid">' + "".join(cards) + "</div>"


def with_gaps(d, xcol):
    """Insert a NaN row wherever consecutive rows skip a calendar season, so lines break at retirements/missing years."""
    d = d.sort_values("PLAYER_SEASON_NUM").reset_index(drop=True)
    rows = []
    for i, row in d.iterrows():
        if i > 0 and row["SEASON_START"] - d.loc[i - 1, "SEASON_START"] > 1:
            blank = {c: np.nan for c in d.columns}
            blank[xcol] = (d.loc[i - 1, xcol] + row[xcol]) / 2
            rows.append(blank)
        rows.append(row.to_dict())
    return pd.DataFrame(rows)


METRICS = {
    "Points per game":              dict(col="PPG",                         kind="num", dec=1, hb=True),
    "True shooting % (TS%)":        dict(col="TS",                          kind="pct", dec=1, hb=True),
    "Effective FG% (eFG%)":         dict(col="EFG",                         kind="pct", dec=1, hb=True),
    "Field goal %":                 dict(col="TOTAL_FG_PCT",                kind="pct", dec=1, hb=True),
    "3-point %":                    dict(col="TOTAL_FG3_PCT",               kind="pct", dec=1, hb=True),
    "Free throw %":                 dict(col="TOTAL_FT_PCT",                kind="pct", dec=1, hb=True),
    "Assists per game":             dict(col="AST_AVE",                     kind="num", dec=1, hb=True),
    "Rebounds per game":            dict(col="REB_AVE",                     kind="num", dec=1, hb=True),
    "Steals per game":              dict(col="STL_AVE",                     kind="num", dec=2, hb=True),
    "Blocks per game":              dict(col="BLK_AVE",                     kind="num", dec=2, hb=True),
    "Turnovers per game":           dict(col="TOV_AVE",                     kind="num", dec=1, hb=False),
    "3-pt attempts per game":       dict(col="FG3A_PG",                     kind="num", dec=1, hb=True),
    "Free-throw attempts per game": dict(col="FTA_PG",                      kind="num", dec=1, hb=True),
    "Games played (regular season)": dict(col="GAMES_PLAYED_REGULAR_SEASON", kind="num", dec=0, hb=True),
    "Regular-season win %":         dict(col="REGULAR_SEASON_WIN_PCT",      kind="pct", dec=1, hb=True),
}


def fmt_val(v, m):
    if pd.isna(v):
        return "–"
    return f"{v * 100:.1f}%" if m["kind"] == "pct" else f"{v:,.{m['dec']}f}"


def hover_fmt(m):
    return ".1%" if m["kind"] == "pct" else f".{m['dec']}f"


def xinfo(xmode):
    if xmode == "Career year":
        return "PLAYER_SEASON_NUM", "Career year (1 = rookie season)"
    return "SEASON_START", "Season start year"


def add_series(fig, p, mkey, xcol, smooth, pos=None, legend=True, star=True, label=True, size=8, awards=False):
    """One player's line for one metric: gap-aware, hollow markers for low-game seasons, star on the career best."""
    m = METRICS[mkey]
    c = m["col"]
    d = SEASON_BY_PLAYER[p].copy()
    d["_RAW"] = d[c]
    d["_Y"] = d[c].rolling(3, center=True, min_periods=1).mean() if smooth else d[c]
    g = with_gaps(d, xcol)
    color = PLAYER_COLORS[p]
    low = g["LOW_GAMES"].eq(True).tolist()
    kw = dict(row=pos[0], col=pos[1]) if pos else {}
    hf = hover_fmt(m)
    cd = g[["SEASON_YEAR", "PLAYER_SEASON_NUM", "GAMES_PLAYED_TOTAL", "_RAW"]].astype(object).values.tolist()
    ht = (f"<b>{p}</b> · %{{customdata[0]}}<br>Career year %{{customdata[1]:.0f}} · games %{{customdata[2]:.0f}}"
          f"<br>{mkey}: %{{y:{hf}}}" + (f"<br>Raw value: %{{customdata[3]:{hf}}}" if smooth else "") + "<extra></extra>")
    if smooth:
        fig.add_trace(go.Scatter(x=g[xcol], y=g["_RAW"], mode="lines", connectgaps=False, hoverinfo="skip",
                                 line=dict(color=color, width=1, dash="dot"), opacity=0.3, showlegend=False,
                                 legendgroup=p), **kw)
    fig.add_trace(go.Scatter(
        x=g[xcol], y=g["_Y"], mode="lines+markers", name=p, legendgroup=p, showlegend=legend, connectgaps=False,
        line=dict(color=color, width=2.5, dash=DASHES[p]),
        marker=dict(symbol=SYMBOLS[p], size=size, color=[THEME["panel"] if lw else color for lw in low],
                    line=dict(color=color, width=2)),
        customdata=cd, hovertemplate=ht), **kw)
    if star:
        q = d[d["QUALIFIED"]]
        idx = q["_Y"].idxmax() if m["hb"] else q["_Y"].idxmin()
        fig.add_trace(go.Scatter(
            x=[d.loc[idx, xcol]], y=[d.loc[idx, "_Y"]], mode="markers+text" if label else "markers",
            marker=dict(symbol="star", size=16, color=color, line=dict(color="#FFFFFF", width=1)),
            text=[f"{'Best' if m['hb'] else 'Lowest'}: {fmt_val(d.loc[idx, '_Y'], m)} ({d.loc[idx, 'SEASON_YEAR']})"] if label else None,
            textposition="top center", textfont=dict(color=color, size=11), hoverinfo="skip", showlegend=False,
            legendgroup=p, cliponaxis=False), **kw)
    if awards:
        for key, sym, sz, lab in (("champion_seasons", "triangle-up", 13, "NBA champion"), ("mvp_seasons", "circle-open", 20, "League MVP")):
            r = d[d["SEASON_YEAR"].isin(season_list(p, key))]
            if len(r):
                fill = "#FFFFFF" if sym == "triangle-up" else "rgba(0,0,0,0)"
                fig.add_trace(go.Scatter(
                    x=r[xcol], y=r["_Y"], mode="markers", showlegend=False, legendgroup=p,
                    marker=dict(symbol=sym, size=sz, color=fill, line=dict(color=color if sym == "triangle-up" else "#FFFFFF", width=2.5)),
                    customdata=r[["SEASON_YEAR"]].astype(object).values.tolist(),
                    hovertemplate=f"<b>{p}</b> · %{{customdata[0]}}<br>{lab} season<extra></extra>"), **kw)


def empty_msg():
    return HTML('<div class="warnbox">Select at least one player to see this view.</div>')

# PART 1 · GOAT CALCULATOR
Decide what "greatest" means to you. Thirteen categories — ten built from the game-log CSVs and three from the external accolades table (marked †) — each made of named sub-metrics. Every sub-metric is scored **0–100 relative to the best of the three players** (`value / best`; for lower-is-better metrics `best / value`). Min-max scaling is deliberately avoided: with only three players it would hand someone a flat 0 in every category.
**Two levels of weights, both on a 1–10 scale (1 = barely matters, 10 = matters most):**
1. **Category sliders** set how much each of the 13 categories counts toward the GOAT score.
2. Open **Fine-tune individual stats** to set the weight of every stat *inside* its category (e.g. TS% vs FG% inside Efficiency).

**Category score** = weighted average of its stats' 0–100 scores. **GOAT score = Σ (category score × category weight share).** Everything — podium, charts, robustness check — updates live as you drag, and the % next to each slider shows its real share of the final score.

**Deliberately not scored:** total points, total games and seasons played. They measure how long a career lasted, not how great it was, so they would hand the longest career an automatic edge. They are still shown in Part 2 as descriptive context, and selections such as All-Star/All-NBA are scored **per season played** by default (`HONORS_AS_RATES`).

In [ ]:
display(HTML('<div class="sec">Meet the contenders</div>' + player_cards_html()))

HONOR_ITEMS = ([("allstar_per_season", "All-Star selections / season", True), ("allnba_first_per_season", "All-NBA 1st Team / season", True),
                ("allnba_per_season", "All-NBA (any team) / season", True), ("alldef_per_season", "All-Defensive Team / season", True)]
               if HONORS_AS_RATES else
               [("allstar_selections", "All-Star selections", True), ("allnba_first", "All-NBA 1st Team selections", True),
                ("allnba_total", "All-NBA selections (any team)", True), ("alldefensive", "All-Defensive selections", True)])

CAT_META = {   # category -> [(column, label, higher_is_better)]   († = external accolades table)
    "Scoring":       [("PTS_AVE", "Career PPG", True), ("PTS_MAX", "Career-high points", True)],
    "Efficiency":    [("TOTAL_FG_PCT", "FG%", True), ("TOTAL_FG3_PCT", "3P%", True), ("TOTAL_FT_PCT", "FT%", True),
                      ("EFG", "eFG%", True), ("TS", "TS%", True)],
    "Playmaking":    [("AST_AVE", "Assists / game", True), ("AST_TOV", "Assist-to-turnover ratio", True)],
    "Rebounding":    [("REB_AVE", "Rebounds / game", True)],
    "Defense":       [("STL_AVE", "Steals / game", True), ("BLK_AVE", "Blocks / game", True)],
    "Ball Security": [("TOV_AVE", "Turnovers / game", False), ("FOULS_AVE", "Fouls / game", False),
                      ("FOULED_OUT_RATE", "Foul-outs per 100 games", False)],
    "Winning":       [("REGULAR_SEASON_WIN_PCT", "Reg-season win %", True), ("PO_SHARE", "Playoff share of games", True),
                      ("PO_SEASON_RATE", "Playoff seasons / seasons", True)],
    "Durability":    [("GP_PER_SEASON", "Reg-season games / season", True)],
    "Peak":          [("PEAK_PPG", "Best-season PPG", True), ("PEAK_TS", "Best-season TS%", True),
                      ("BEST3_PPG", "Best 3-season PPG", True)],
    "Consistency":   [("PPG_CV", "Season-to-season PPG variation", False), ("TS_STD", "Season-to-season TS% spread", False)],
    "Championships": [("championships", "NBA titles", True), ("finals_mvp", "Finals MVP awards", True),
                      ("finals_win_pct", "Finals win rate", True)],
    "Individual Awards": [("mvp", "League MVP awards", True), ("dpoy", "Defensive Player of the Year", True),
                          ("scoring_titles", "Scoring titles", True), ("allstar_mvp", "All-Star Game MVPs", True)],
    "Honors":        HONOR_ITEMS,
}
EXTERNAL_CATS = {"Championships", "Individual Awards", "Honors"}
CATS = list(CAT_META)


def normalise(s, higher_better=True):
    s = s.astype(float)
    if higher_better:
        return (s / s.max() if s.max() > 0 else s * 0 + 1) * 100      # nobody has any -> everyone ties at 100
    return (s.min() / s) * 100


_sub = {(cat, label): normalise(CAREER_X[col], hb) for cat, items in CAT_META.items() for col, label, hb in items}
SUBSCORES = pd.DataFrame(_sub)
SUBSCORES.columns = pd.MultiIndex.from_tuples(SUBSCORES.columns)

PRESETS = {
    "Balanced (equal weights)": {c: WEIGHT_DEFAULT for c in CATS},
    "Pure Scorer":        dict(Scoring=10, Peak=7, Efficiency=6),
    "Winner":             dict(Championships=10, Winning=7, Peak=4, Consistency=3),
    "Trophy Case":        dict(Championships=10, **{"Individual Awards": 10}, Honors=7, Peak=4),
    "Iron Man":           dict(Durability=10, Consistency=6, Honors=5, Winning=4),
    "Efficiency Nerd":    dict(Efficiency=10, **{"Ball Security": 7}, Consistency=5, Scoring=3),
    "Two-Way Player":     dict(Defense=10, Scoring=6, Rebounding=5, Playmaking=5, Efficiency=4, **{"Ball Security": 3}),
    "Peak Performance":   dict(Peak=10, Scoring=6, Efficiency=4),
}
PRESETS = {k: (v if k.startswith("Balanced") else {c: v.get(c, max(WEIGHT_MIN, 1)) for c in CATS}) for k, v in PRESETS.items()}
CAT_COLORS = ["#C8102E", "#3B6FD4", "#FDBB30", "#2FBF9B", "#8A5CC9", "#FF7A45", "#5BC0EB", "#E26A8D", "#9BC53D", "#B0B7C8",
              "#F4D35E", "#7A9E9F", "#D1495B"]
assert len(CAT_COLORS) >= len(CATS)
HEAT_H = 90 + 34 * len(CATS)


def category_scores(sub_w=None):
    """Category score (0-100) per player = weighted mean of that category's stat scores.
    sub_w: {(category, stat label): weight}. None means every stat counts equally."""
    out = {}
    for cat, items in CAT_META.items():
        w = np.array([1.0 if sub_w is None else float(sub_w.get((cat, lab), 0)) for _, lab, _ in items])
        if w.sum() <= 0:
            w = np.ones(len(items))
        w = w / w.sum()
        out[cat] = sum(SUBSCORES[(cat, lab)] * wi for (_, lab, _), wi in zip(items, w))
    return pd.DataFrame(out).loc[PLAYERS, CATS]


CAT_SCORES = category_scores()          # reference scores with equal stat weights


def goat_scores(weights, cat_scores=None):
    """weights: {category: weight}. Returns (total, contributions, normalised category weights, warning)."""
    cs = CAT_SCORES if cat_scores is None else cat_scores
    w = pd.Series({c: float(weights.get(c, 0)) for c in CATS}).clip(lower=0)
    warn = None
    if w.sum() == 0:
        w[:] = 1.0
        warn = "All weights are 0 — falling back to equal weights."
    w = w / w.sum()
    contrib = cs.mul(w, axis=1)
    total = contrib.sum(axis=1).sort_values(ascending=False)
    return total, contrib.loc[total.index], w, warn


def effective_weights(cat_w, sub_w):
    """Share of the final score controlled by each category and each stat (they sum to 100%)."""
    cw = pd.Series({c: max(float(cat_w.get(c, 0)), 0.0) for c in CATS})
    cw = cw / cw.sum() if cw.sum() > 0 else cw * 0 + 1 / len(CATS)
    within, eff = {}, {}
    for cat, items in CAT_META.items():
        w = np.array([max(float(sub_w.get((cat, lab), 0)), 0.0) for _, lab, _ in items])
        w = w / w.sum() if w.sum() > 0 else np.ones(len(items)) / len(items)
        for (_, lab, _), wi in zip(items, w):
            within[(cat, lab)] = wi
            eff[(cat, lab)] = cw[cat] * wi
    return cw, within, eff


# ---- figures ------------------------------------------------------------------------------
def fig_contrib(total, contrib):
    order = list(total.index)[::-1]                    # winner on top of a horizontal bar chart
    fig = go.Figure()
    for cat, color in zip(CATS, CAT_COLORS):
        fig.add_trace(go.Bar(
            y=order, x=[contrib.loc[p, cat] for p in order], orientation="h", name=cat, marker_color=color,
            hovertemplate="<b>%{y}</b><br>" + cat + ": %{x:.1f} pts<extra></extra>"))
    for p in order:
        fig.add_annotation(x=total[p], y=p, text=f"<b>{total[p]:.1f}</b>", showarrow=False, xanchor="left",
                           xshift=8, font=dict(color=PLAYER_COLORS[p], size=15))
    fig.update_layout(barmode="stack")
    style(fig, "Why they win: weighted points by category", height=HEAT_H, margin=dict(l=110, r=40, t=70, b=175))
    fig.update_layout(legend=dict(orientation="h", x=0, xanchor="left", y=-0.26, yanchor="top", font=dict(size=10)))
    fig.update_xaxes(title="GOAT score (0–100)", range=[0, 112])
    fig.update_yaxes(tickfont=dict(size=13))
    return fig


def fig_radar(cs=None):
    cs = CAT_SCORES if cs is None else cs
    lo = max(0, int(np.floor((cs.min().min() - 10) / 10) * 10))
    fig = go.Figure()
    for p in PLAYERS:
        r = cs.loc[p].tolist()
        fig.add_trace(go.Scatterpolar(
            r=r + r[:1], theta=CATS + CATS[:1], name=p, fill="toself", fillcolor=rgba(PLAYER_COLORS[p], 0.16),
            line=dict(color=PLAYER_COLORS[p], width=2.5, dash=DASHES[p]), hovertemplate="%{theta}: %{r:.1f}<extra>" + p + "</extra>"))
    style(fig, "Category scores (0–100, relative to the best)", height=HEAT_H, margin=dict(l=60, r=60, t=70, b=40))
    fig.update_layout(polar=dict(
        bgcolor=THEME["panel"],
        radialaxis=dict(range=[lo, 100], gridcolor=THEME["grid"], linecolor=THEME["grid"], tickfont=dict(color=THEME["muted"], size=10)),
        angularaxis=dict(gridcolor=THEME["grid"], linecolor=THEME["grid"], tickfont=dict(size=11))))
    return fig


def fig_heat(w, cs=None):
    cs = CAT_SCORES if cs is None else cs
    z = cs.T.loc[CATS, PLAYERS]
    labels = [f"{c}{chr(8224) if c in EXTERNAL_CATS else chr(32)}  ({w[c] * 100:.0f}%)" for c in CATS]
    fig = go.Figure(go.Heatmap(
        z=z.values, x=PLAYERS, y=labels, zmin=max(0, np.floor(z.values.min() / 5) * 5 - 5), zmax=100, xgap=3, ygap=3,
        colorscale=[[0, THEME["panel_alt"]], [1, THEME["nba_blue_light"]]], showscale=False,
        text=np.round(z.values, 1), texttemplate="%{text}", textfont=dict(color=THEME["text"], size=13),
        hovertemplate="%{y}<br>%{x}: %{z:.1f}<extra></extra>"))
    for i, c in enumerate(CATS):
        best = z.loc[c].max()
        for j, p in enumerate(PLAYERS):
            if z.loc[c, p] >= best - 1e-9:
                fig.add_shape(type="rect", x0=j - 0.5, x1=j + 0.5, y0=i - 0.5, y1=i + 0.5, xref="x", yref="y",
                              line=dict(color=PLAYER_COLORS[p], width=3))
    style(fig, "Who wins each category (winner outlined) · label shows its share of the score", height=HEAT_H,
          legend=False, margin=dict(l=215, r=20, t=70, b=40))
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_xaxes(side="top", showgrid=False)
    return fig


# ---- weight sweep: vary ONE slider from min to max, everything else fixed -------------------------
def sweep_options():
    opts = [(f"Category · {c}{' †' if c in EXTERNAL_CATS else ''}", ("cat", c)) for c in CATS]
    opts += [(f"{c} ▸ {lab}", ("sub", c, lab)) for c, items in CAT_META.items() for _, lab, _ in items]
    return opts


def sweep_frame(target, cat_w, sub_w, grid):
    rows = []
    for v in grid:
        cw, sw = dict(cat_w), dict(sub_w)
        if target[0] == "cat":
            cw[target[1]] = v
        else:
            sw[(target[1], target[2])] = v
        t, _, _, _ = goat_scores(cw, category_scores(sw))
        rows.append(t.reindex(PLAYERS))
    return pd.DataFrame(rows, index=grid)


def fig_sens(target, cat_w, sub_w):
    grid = np.linspace(WEIGHT_MIN, WEIGHT_MAX, 37)
    df = sweep_frame(target, cat_w, sub_w, grid)
    label = target[1] if target[0] == "cat" else f"{target[1]} ▸ {target[2]}"
    cur = cat_w[target[1]] if target[0] == "cat" else sub_w[(target[1], target[2])]
    fig = go.Figure()
    for p in PLAYERS:
        fig.add_trace(go.Scatter(x=grid, y=df[p], mode="lines", name=p, line=dict(color=PLAYER_COLORS[p], width=3, dash=DASHES[p]),
                                 hovertemplate=f"<b>{p}</b><br>{label} weight: %{{x:.1f}}<br>GOAT score: %{{y:.1f}}<extra></extra>"))
    leader = df.idxmax(axis=1)
    for i in range(1, len(grid)):
        if leader.iloc[i] != leader.iloc[i - 1]:
            fig.add_vline(x=grid[i], line=dict(color=THEME["muted"], width=1, dash="dot"))
            fig.add_annotation(x=grid[i], y=1.0, yref="paper", yanchor="bottom", showarrow=False,
                               text=f"{leader.iloc[i]} takes #1", font=dict(color=PLAYER_COLORS[leader.iloc[i]], size=11))
    fig.add_vline(x=cur, line=dict(color=THEME["nba_red"], width=2, dash="dash"))
    fig.add_annotation(x=cur, y=1.0, yref="paper", yanchor="bottom", yshift=18, showarrow=False,
                       text=f"your weight: {cur}", font=dict(color=THEME["nba_red"], size=11))
    style(fig, f"Weight sweep: what if “{label}” mattered more or less?", height=420, margin=dict(l=60, r=30, t=100, b=60))
    fig.update_xaxes(title=f"Weight given to {label} ({WEIGHT_MIN}–{WEIGHT_MAX}); every other slider stays where you set it",
                     range=[WEIGHT_MIN, WEIGHT_MAX], dtick=1)
    fig.update_yaxes(title="GOAT score")
    return fig, leader


# ---- trophy case ----------------------------------------------------------------------------
TROPHIES = [("championships", "NBA titles"), ("finals_mvp", "Finals MVP"), ("mvp", "League MVP"), ("dpoy", "Defensive POY"),
            ("scoring_titles", "Scoring titles"), ("allstar_mvp", "All-Star MVP"), ("olympic_gold", "Olympic gold")]


def fig_trophies():
    right_title = ("Selections as a share of seasons played (what the calculator scores)" if HONORS_AS_RATES
                   else "Selections: career totals (what the calculator scores)")
    fig = make_subplots(rows=1, cols=2, column_widths=[0.56, 0.44], subplot_titles=["Major trophies (career totals)", right_title],
                        horizontal_spacing=0.09)
    fig.update_annotations(font=dict(color=THEME["text"], size=13))
    for p in PLAYERS:
        v = [CAREER_X.loc[p, c] for c, _ in TROPHIES]
        fig.add_trace(go.Bar(x=[l for _, l in TROPHIES], y=v, name=p, legendgroup=p, marker_color=PLAYER_COLORS[p],
                             text=[f"{x:.0f}" for x in v], textposition="outside",
                             hovertemplate=f"<b>{p}</b><br>%{{x}}: %{{y:.0f}}<extra></extra>"), row=1, col=1)
        hv = [CAREER_X.loc[p, c] for c, _, _ in HONOR_ITEMS]
        fig.add_trace(go.Bar(x=[l.replace(" / season", "") for _, l, _ in HONOR_ITEMS], y=hv, name=p, legendgroup=p, showlegend=False,
                             marker_color=PLAYER_COLORS[p], text=[f"{x:.0%}" if HONORS_AS_RATES else f"{x:.0f}" for x in hv],
                             textposition="outside",
                             hovertemplate=f"<b>{p}</b><br>%{{x}}: " + ("%{y:.1%}" if HONORS_AS_RATES else "%{y:.0f}") + "<extra></extra>"),
                      row=1, col=2)
    style(fig, None, height=450, margin=dict(l=55, r=20, t=95, b=60))
    fig.update_layout(barmode="group", bargap=0.22, legend=dict(orientation="h", x=0.5, xanchor="center", y=1.2, yanchor="bottom"))
    top_l = max(CAREER_X[c].max() for c, _ in TROPHIES) * 1.18
    top_r = max(CAREER_X[c].max() for c, _, _ in HONOR_ITEMS) * 1.18
    fig.update_yaxes(range=[0, top_l], row=1, col=1)
    fig.update_yaxes(range=[0, top_r], row=1, col=2, tickformat=".0%" if HONORS_AS_RATES else None)
    fig.update_xaxes(tickfont=dict(size=10))
    return fig


def trophy_caption():
    return (f"† Accolade counts are compiled by hand (section 1b, as of {ACC_AS_OF}); they are not in the game-log CSVs. "
            "Finals win rate = titles ÷ Finals appearances. "
            + ("The right panel shows selections divided by seasons played, so a longer career is not rewarded just for being longer."
               if HONORS_AS_RATES else "Raw selection totals favour longer careers; set HONORS_AS_RATES = True to score them per season."))


# ---- Monte-Carlo robustness: random CATEGORY weights, using your current stat-level weights ---------
_rng = np.random.default_rng(MC_SEED)
MC_W = _rng.dirichlet(np.ones(len(CATS)), size=MC_SIMULATIONS)


def mc_shares(cs=None):
    cs = CAT_SCORES if cs is None else cs
    scores = MC_W @ cs.loc[PLAYERS, CATS].values.T
    ranks = (-scores).argsort(axis=1).argsort(axis=1) + 1
    return (pd.Series((ranks == 1).mean(axis=0) * 100, index=PLAYERS),
            pd.Series((ranks == len(PLAYERS)).mean(axis=0) * 100, index=PLAYERS))


MC_FIRST, MC_LAST = mc_shares()


def fig_mc(first=None, last=None):
    first = MC_FIRST if first is None else first
    last = MC_LAST if last is None else last
    fig = go.Figure()
    fig.add_trace(go.Bar(x=PLAYERS, y=first.values, name="Ranks #1 (GOAT)", marker=dict(color=[PLAYER_COLORS[p] for p in PLAYERS]),
                         text=[f"{v:.1f}%" for v in first.values], textposition="outside",
                         hovertemplate="<b>%{x}</b> ranks #1 in %{y:.1f}% of scenarios<extra></extra>"))
    fig.add_trace(go.Bar(x=PLAYERS, y=last.values, name="Ranks last (#3)",
                         marker=dict(color=[PLAYER_COLORS[p] for p in PLAYERS], pattern=dict(shape="/", fgcolor=THEME["panel"], solidity=0.55)),
                         text=[f"{v:.1f}%" for v in last.values], textposition="outside",
                         hovertemplate="<b>%{x}</b> ranks last in %{y:.1f}% of scenarios<extra></extra>"))
    style(fig, f"Robustness: {MC_SIMULATIONS:,} random category weightings", height=HEAT_H, margin=dict(l=60, r=30, t=90, b=60))
    fig.update_layout(barmode="group", bargap=0.25)
    fig.update_yaxes(title="% of scenarios", range=[0, max(first.max(), last.max()) * 1.2 + 5])
    return fig


def mc_caption(first=None):
    first = MC_FIRST if first is None else first
    best = first.idxmax()
    s = ", ".join(f"{p} {v:.0f}%" for p, v in first.sort_values(ascending=False).items())
    return (f"Each scenario draws a random weight for every one of the {len(CATS)} categories (Dirichlet(1), seed {MC_SEED}) and scores them with "
            f"<b>your current stat-level weights</b>. <b style='color:{PLAYER_COLORS[best]}'>{best}</b> finishes first most often. #1 share: {s}. "
            "A dominant share means the verdict barely depends on category weights; a split means it does.")


def podium_html(total, contrib, warn):
    ordr = list(total.index)
    cards = []
    for i, p in enumerate(ordr):
        gap = "Leader" if i == 0 else f"{total.iloc[0] - total[p]:.1f} pts behind #1"
        cards.append(f'<div class="pod{" first" if i == 0 else ""}" style="--c:{PLAYER_COLORS[p]}">'
                     f'<div class="rank">#{i + 1}</div><div class="name">{p}</div>'
                     f'<div class="score">{total[p]:.1f}</div><div class="gap">{gap}</div></div>')
    win, run = ordr[0], ordr[1]
    margin = total[win] - total[run]
    delta = contrib.loc[win] - contrib.loc[run]
    driver = delta.idxmax()
    tight = " — a razor-thin margin that a small slider change could flip." if margin < 1 else "."
    verdict = (f'<b style="color:{PLAYER_COLORS[win]}">{win}</b> is your GOAT with <b>{total[win]:.1f}</b>, '
               f'<b>{margin:.1f}</b> points ahead of {run}. Biggest edge over {run}: <b>{driver}</b> '
               f'(+{delta[driver]:.1f} weighted points){tight}')
    out = ""
    if warn:
        out += f'<div class="warnbox">{warn}</div>'
    return out + '<div class="podium">' + "".join(cards) + "</div>" + f'<div class="verdict">{verdict}</div>'

In [ ]:
class LiveChart:
    """A Plotly chart that updates IN PLACE (FigureWidget) when anywidget is available; otherwise it redraws inside an Output widget."""

    def __init__(self, fig, layout_keys=()):
        self.keys = layout_keys
        if LIVE_PLOTS:
            self.w = go.FigureWidget(fig)
            self.view = W.Box([self.w], layout=W.Layout(width="100%"))
        else:
            self.out = W.Output()
            self.view = self.out
            self.update(fig)

    def update(self, fig):
        if LIVE_PLOTS:
            with self.w.batch_update():
                for old, new in zip(self.w.data, fig.data):
                    props = new.to_plotly_json()
                    props.pop("type", None)
                    old.update(props)
                for k in self.keys:
                    self.w.layout[k] = fig.layout[k]
        else:
            with self.out:
                clear_output(wait=True)
                show(fig)


def build_goat_calculator():
    ext = lambda c: " †" if c in EXTERNAL_CATS else ""
    slider = lambda desc, dw, width: W.IntSlider(value=WEIGHT_DEFAULT, min=WEIGHT_MIN, max=WEIGHT_MAX, step=1, description=desc,
                                                continuous_update=True, style={"description_width": dw},
                                                layout=W.Layout(width=width))
    cat_sl = {c: slider(c + ext(c), "150px", "400px") for c in CATS}
    cat_ro = {c: W.HTML(layout=W.Layout(width="70px")) for c in CATS}
    sub_sl = {(c, lab): slider(lab, "235px", "480px") for c, items in CAT_META.items() for _, lab, _ in items}
    sub_ro = {k: W.HTML(layout=W.Layout(width="190px")) for k in sub_sl}
    CUSTOM = "Custom (your sliders)"
    preset_dd = W.Dropdown(options=[CUSTOM] + list(PRESETS), value="Balanced (equal weights)", description="Preset",
                           style={"description_width": "55px"}, layout=W.Layout(width="330px"))
    reset_all = W.Button(description="Reset all", icon="refresh", layout=W.Layout(width="110px"))
    reset_stats = W.Button(description="Reset stat weights", icon="undo", layout=W.Layout(width="170px"))
    sens_dd = W.Dropdown(options=sweep_options(), value=("cat", "Scoring"), description="Sweep",
                         style={"description_width": "50px"}, layout=W.Layout(width="560px"))
    podium = W.HTML()
    mc_cap, sens_cap = W.HTML(), W.HTML()
    trophy_out = W.Output()
    busy = {"on": False, "preset": False}

    cat_w = lambda: {c: s.value for c, s in cat_sl.items()}
    sub_w = lambda: {k: s.value for k, s in sub_sl.items()}

    # ---- first draw (all weights at the default) builds the charts; later changes update them in place
    cw0, sw0 = cat_w(), sub_w()
    cs0 = category_scores(sw0)
    total0, contrib0, wn0, warn0 = goat_scores(cw0, cs0)
    first0, last0 = mc_shares(cs0)
    charts = {
        "contrib": LiveChart(fig_contrib(total0, contrib0), ("annotations",)),
        "radar":   LiveChart(fig_radar(cs0), ("polar",)),
        "heat":    LiveChart(fig_heat(wn0, cs0), ("shapes",)),
        "mc":      LiveChart(fig_mc(first0, last0), ("yaxis",)),
        "sens":    LiveChart(fig_sens(sens_dd.value, cw0, sw0)[0], ("shapes", "annotations", "title", "xaxis")),
    }

    def render(_=None):
        if busy["on"]:
            return
        cw, sw = cat_w(), sub_w()
        cs = category_scores(sw)
        total, contrib, wn, warn = goat_scores(cw, cs)
        _, within, eff = effective_weights(cw, sw)
        for c in CATS:
            cat_ro[c].value = f"<b style='color:{THEME['text']}'>{wn[c] * 100:.0f}%</b>"
        for k in sub_sl:
            sub_ro[k].value = (f"<span style='color:{THEME['muted']};font-size:12px'><b style='color:{THEME['text']}'>{eff[k] * 100:.1f}%</b> of score"
                               f" · {within[k] * 100:.0f}% of category</span>")
        podium.value = podium_html(total, contrib, warn)
        charts["contrib"].update(fig_contrib(total, contrib))
        charts["radar"].update(fig_radar(cs))
        charts["heat"].update(fig_heat(wn, cs))
        first, last = mc_shares(cs)
        charts["mc"].update(fig_mc(first, last))
        mc_cap.value = f'<div class="cap">{mc_caption(first)}</div>'
        render_sens()

    def render_sens(_=None):
        if busy["on"]:
            return
        cw, sw = cat_w(), sub_w()
        fig, leader = fig_sens(sens_dd.value, cw, sw)
        charts["sens"].update(fig)
        flips = int((leader != leader.shift()).sum() - 1)
        txt = (f"The #1 spot never changes as this weight moves from {WEIGHT_MIN} to {WEIGHT_MAX}." if flips == 0 else
               f"The #1 spot changes {flips} time(s) as this weight moves from {WEIGHT_MIN} to {WEIGHT_MAX}; dotted lines mark the crossovers.")
        sens_cap.value = f'<div class="cap">{txt} The dashed red line is where your slider currently sits. Pick any category or individual stat in the dropdown.</div>'

    def reset_stat_sliders():
        for s_ in sub_sl.values():
            s_.value = WEIGHT_DEFAULT

    def apply_preset(name):
        busy["on"] = True
        for c in CATS:
            cat_sl[c].value = PRESETS[name][c]
        reset_stat_sliders()
        busy["on"] = False
        render()

    def do_reset_all(_b):
        busy["preset"] = True
        preset_dd.value = "Balanced (equal weights)"
        busy["preset"] = False
        apply_preset("Balanced (equal weights)")

    def on_slider(_=None):
        if not busy["on"] and preset_dd.value != CUSTOM:       # any manual move switches the dropdown to "Custom"
            busy["preset"] = True
            preset_dd.value = CUSTOM
            busy["preset"] = False
        render()

    def on_preset(change):
        if not busy["preset"] and change["new"] != CUSTOM:
            apply_preset(change["new"])

    def do_reset_stats(_b):
        busy["on"] = True
        reset_stat_sliders()
        busy["on"] = False
        on_slider()

    for s_ in list(cat_sl.values()) + list(sub_sl.values()):
        s_.observe(on_slider, names="value")
    preset_dd.observe(on_preset, names="value")
    reset_all.on_click(do_reset_all)
    reset_stats.on_click(do_reset_stats)
    sens_dd.observe(render_sens, names="value")

    with trophy_out:
        show(fig_trophies())
        display(caption(trophy_caption()))

    # ---- layout
    half = (len(CATS) + 1) // 2
    cat_box = W.HBox([W.VBox([W.HBox([cat_sl[c], cat_ro[c]]) for c in CATS[:half]]),
                      W.VBox([W.HBox([cat_sl[c], cat_ro[c]]) for c in CATS[half:]])])
    panels = [W.VBox([W.HBox([sub_sl[(c, lab)], sub_ro[(c, lab)]]) for _, lab, _ in items]) for c, items in CAT_META.items()]
    fine = W.Accordion(children=panels, selected_index=None)
    fine.titles = tuple(f"{c}{ext(c)}  —  {len(items)} stat{'s' if len(items) > 1 else ''}" for c, items in CAT_META.items())
    how = "".join(
        f"<tr><td><b>{cat}{ext(cat)}</b></td><td>" + " · ".join(f"{lab}{'' if hb else ' <i>(lower is better)</i>'}" for _, lab, hb in items) + "</td></tr>"
        for cat, items in CAT_META.items())
    acc = W.Accordion(children=[W.HTML(f'<table class="methtbl"><tr><th>Category</th><th>Stats (each scored 0–100 vs the best of the three)</th></tr>{how}</table>'
                                       '<div class="cap">† From the external accolades table (section 1b), not the CSVs. Total points, total games and seasons played are intentionally excluded.</div>')],
                      selected_index=None)
    acc.titles = ("How scores are computed",)
    hint = W.HTML(f'<div class="cap">Weights run <b>{WEIGHT_MIN}–{WEIGHT_MAX}</b> ({WEIGHT_MIN} = barely matters, {WEIGHT_MAX} = matters most). '
                  'The % beside each category slider is its share of the final score. Open <b>Fine-tune individual stats</b> to weight each stat inside its category; '
                  'its readout shows the stat\'s share of the whole score. Everything below updates as you drag.</div>')
    half_box = lambda ch: W.Box([ch.view], layout=W.Layout(width="49%"))
    root = W.VBox([
        W.HTML('<div class="sec">1 · Set your weights</div>'), hint,
        W.HBox([preset_dd, reset_all, reset_stats]),
        W.HTML('<div class="cap" style="margin-top:8px"><b>Category weights</b></div>'), cat_box,
        fine, acc,
        W.HTML('<div class="sec">2 · The verdict (live)</div>'), podium,
        W.HBox([half_box(charts["contrib"]), half_box(charts["radar"])]),
        W.HTML('<div class="sec">3 · The trophy case †</div>'), trophy_out,
        W.HTML('<div class="sec">4 · Category winners &amp; robustness (live)</div>'),
        W.HBox([half_box(charts["heat"]), W.VBox([charts["mc"].view, mc_cap], layout=W.Layout(width="49%"))]),
        W.HTML('<div class="sec">5 · Weight sweep: how sensitive is the result to one slider?</div>'),
        sens_dd, charts["sens"].view, sens_cap,
    ])
    root.add_class("goat-root")
    root.refs = dict(cat=cat_sl, sub=sub_sl, preset=preset_dd, charts=charts, podium=podium, sens=sens_dd,
                     reset_all=reset_all, reset_stats=reset_stats, render=render)
    render()
    return root


GOAT_UI = build_goat_calculator()
display(GOAT_UI)

# PART 2 · CAREER TREND ANALYSIS WITH INSIGHTS
Nine linked views plus nine auto-generated insight cards. Use the controls to pick players, switch between **career year** (aligns rookies at year 1) and **calendar season**, and smooth the lines with a 3-season rolling average.
Chart conventions: ★ = career best · hollow marker = low-game season (< 30 games) · broken line = a calendar gap in the data (e.g. retirement).

In [ ]:
# ---- Part 2 figure builders -----------------------------------------------------------------
def fig_trend(mkey, players, xmode, smooth, awards=True):
    xcol, xtitle = xinfo(xmode)
    fig = go.Figure()
    for p in players:
        add_series(fig, p, mkey, xcol, smooth, awards=awards)
    style(fig, f"{mkey} by season", height=470, margin=dict(l=65, r=30, t=75, b=55))
    fig.update_xaxes(title=xtitle, dtick=1 if xmode == "Career year" else 2)
    fig.update_yaxes(title=mkey, tickformat=".0%" if METRICS[mkey]["kind"] == "pct" else None)
    fig.update_layout(hovermode="closest")
    return fig


SMALL_MULT = ["Points per game", "True shooting % (TS%)", "Assists per game", "Rebounds per game", "Turnovers per game", "Steals per game"]


def fig_small_multiples(players, xmode, smooth):
    xcol, xtitle = xinfo(xmode)
    fig = make_subplots(rows=2, cols=3, subplot_titles=SMALL_MULT, horizontal_spacing=0.06, vertical_spacing=0.16)
    fig.update_annotations(font=dict(color=THEME["text"], size=13))
    for k, mk in enumerate(SMALL_MULT):
        r, c = k // 3 + 1, k % 3 + 1
        for p in players:
            add_series(fig, p, mk, xcol, smooth, pos=(r, c), legend=(k == 0), label=False, size=5)
        if METRICS[mk]["kind"] == "pct":
            fig.update_yaxes(tickformat=".0%", row=r, col=c)
    style(fig, None, height=660, margin=dict(l=50, r=20, t=90, b=50))
    fig.update_layout(legend=dict(orientation="h", x=0.5, xanchor="center", y=1.1, yanchor="bottom"))
    fig.update_xaxes(title=xtitle, row=2)
    return fig


HEAT_METRICS = [("PPG", "Points"), ("TS", "TS%"), ("AST_AVE", "Assists"), ("REB_AVE", "Rebounds"), ("STL_AVE", "Steals"),
                ("BLK_AVE", "Blocks"), ("TOV_AVE", "Turnovers"), ("FG3A_PG", "3PA / game"), ("GAMES_PLAYED_REGULAR_SEASON", "Games (RS)")]


def fig_season_heatmap(players):
    n = len(players)
    years = list(range(1, int(SEASON["PLAYER_SEASON_NUM"].max()) + 1))
    fig = make_subplots(rows=n, cols=1, subplot_titles=players, vertical_spacing=0.09 if n > 1 else 0.1)
    fig.update_annotations(font=dict(size=14))
    for i, p in enumerate(players, 1):
        fig.layout.annotations[i - 1].font.color = PLAYER_COLORS[p]
        d = SEASON_BY_PLAYER[p].set_index("PLAYER_SEASON_NUM")
        z, raw = [], []
        for col, _ in HEAT_METRICS:
            s = d[col].reindex(years)
            z.append(((s - d[col].mean()) / d[col].std(ddof=0)).tolist())
            raw.append(s.tolist())
        fig.add_trace(go.Heatmap(
            z=z, x=years, y=[lab for _, lab in HEAT_METRICS], customdata=raw, zmin=-2.5, zmax=2.5, xgap=2, ygap=2,
            colorscale=[[0, "#2A5BD7"], [0.5, THEME["panel_alt"]], [1, THEME["nba_red"]]],
            showscale=(i == 1), colorbar=dict(title="z-score", thickness=12, len=0.45, y=0.85, tickfont=dict(size=10)),
            hovertemplate="Career year %{x} · %{y}<br>Value: %{customdata:.2f}<br>z-score vs own career: %{z:+.2f}<extra></extra>"),
            row=i, col=1)
    style(fig, None, height=80 + 235 * n, legend=False, margin=dict(l=90, r=60, t=50, b=50))
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_xaxes(showgrid=False, dtick=1)
    fig.update_xaxes(title="Career year", row=n, col=1)
    return fig


CUM_SPECS = [("PTS_TOTAL", "Cumulative points", [10_000, 20_000, 30_000, 40_000, 50_000]),
             ("GAMES_PLAYED_TOTAL", "Cumulative games (RS + PO)", [500, 1000, 1500]),
             ("TOTAL_FGM", "Cumulative field goals made", [5_000, 10_000, 15_000])]


def milestones(p, col, thresholds):
    d = SEASON_BY_PLAYER[p]
    cum = d[col].cumsum().to_numpy(dtype=float)
    gcum = d["GAMES_PLAYED_TOTAL"].cumsum().to_numpy(dtype=float)
    yrs = d["PLAYER_SEASON_NUM"].to_numpy(dtype=float)
    out = []
    for th in thresholds:
        if not (cum >= th).any():
            continue
        i = int(np.argmax(cum >= th))
        pc, py, pg = (cum[i - 1], yrs[i - 1], gcum[i - 1]) if i > 0 else (0.0, 0.0, 0.0)
        f = (th - pc) / (cum[i] - pc)
        out.append(dict(th=th, year=py + f * (yrs[i] - py), games=pg + f * (gcum[i] - pg)))
    return out


def fig_cumulative(players):
    fig = make_subplots(rows=1, cols=3, subplot_titles=[s[1] for s in CUM_SPECS], horizontal_spacing=0.07)
    fig.update_annotations(font=dict(color=THEME["text"], size=13))
    for k, (col, _, ths) in enumerate(CUM_SPECS, 1):
        for p in players:
            d = SEASON_BY_PLAYER[p]
            x = [0] + d["PLAYER_SEASON_NUM"].tolist()
            y = [0] + d[col].cumsum().tolist()
            fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=p, legendgroup=p, showlegend=(k == 1),
                                     line=dict(color=PLAYER_COLORS[p], width=3, dash=DASHES[p]),
                                     hovertemplate=f"<b>{p}</b><br>Career year %{{x}}<br>%{{y:,.0f}}<extra></extra>"), row=1, col=k)
            ms = milestones(p, col, ths)
            if ms:
                fig.add_trace(go.Scatter(
                    x=[m["year"] for m in ms], y=[m["th"] for m in ms], mode="markers+text", showlegend=False, legendgroup=p,
                    marker=dict(symbol="diamond", size=10, color=PLAYER_COLORS[p], line=dict(color="#FFFFFF", width=1)),
                    text=[f"{m['th'] / 1000:g}K" if m["th"] >= 5000 else f"{m['th']:,}" for m in ms], textposition="bottom right",
                    textfont=dict(size=10, color=PLAYER_COLORS[p]),
                    customdata=[[m["games"]] for m in ms],
                    hovertemplate=f"<b>{p}</b> reached %{{y:,.0f}}<br>≈ career year %{{x:.1f}} · ≈ %{{customdata[0]:.0f}} games<extra></extra>"),
                    row=1, col=k)
    style(fig, None, height=430, margin=dict(l=60, r=20, t=70, b=55))
    fig.update_layout(legend=dict(orientation="h", x=0.5, xanchor="center", y=1.12, yanchor="bottom"))
    fig.update_xaxes(title="Career year")
    fig.update_yaxes(tickformat=",")
    return fig


def cumulative_caption(players):
    bits = []
    for th in (10_000, 20_000, 30_000):
        reach = {}
        for p in players:
            m = milestones(p, "PTS_TOTAL", [th])
            if m:
                reach[p] = m[0]
        if reach:
            w = min(reach, key=lambda p: reach[p]["games"])
            bits.append(f"<b>{th // 1000}K points:</b> <span style='color:{PLAYER_COLORS[w]}'>{w}</span> in ≈{reach[w]['games']:.0f} games (≈ career year {reach[w]['year']:.1f})")
    return ("Diamonds mark milestone crossings (linearly interpolated within the season; points include playoffs). Fastest to " + " · ".join(bits) + "."
            if bits else "Diamonds mark milestone crossings.")


def fig_scatter(players, animate=False):
    gmin, gmax = SEASON["GAMES_PLAYED_TOTAL"].min(), SEASON["GAMES_PLAYED_TOTAL"].max()

    def mk(p, upto):
        s = SEASON_BY_PLAYER[p]
        s = s[s["PLAYER_SEASON_NUM"] <= upto]
        size = 9 + 24 * (s["GAMES_PLAYED_TOTAL"] - gmin) / (gmax - gmin)
        return go.Scatter(
            x=s["PPG"], y=s["TS"], mode="lines+markers", name=p, legendgroup=p,
            line=dict(color=rgba(PLAYER_COLORS[p], 0.35), width=1.2),
            marker=dict(size=size, color=PLAYER_COLORS[p], opacity=0.85, symbol=SYMBOLS[p], line=dict(color="#FFFFFF", width=0.8)),
            customdata=s[["SEASON_YEAR", "PLAYER_SEASON_NUM", "GAMES_PLAYED_TOTAL"]].astype(object).values.tolist(),
            hovertemplate=f"<b>{p}</b> · %{{customdata[0]}} (career yr %{{customdata[1]:.0f}})<br>PPG %{{x:.1f}} · TS% %{{y:.1%}}"
                          "<br>Games %{customdata[2]:.0f}<extra></extra>")

    maxk = int(SEASON["PLAYER_SEASON_NUM"].max())
    if animate:
        fig = go.Figure(data=[mk(p, 1) for p in players],
                        frames=[go.Frame(data=[mk(p, k) for p in players], name=str(k)) for k in range(1, maxk + 1)])
    else:
        fig = go.Figure(data=[mk(p, maxk) for p in players])
    style(fig, "Efficiency vs volume (bubble size = games played)", height=520 if animate else 480,
          margin=dict(l=65, r=30, t=75, b=125 if animate else 55))
    fig.update_xaxes(title="Points per game (RS + PO)", range=[SEASON["PPG"].min() - 2, SEASON["PPG"].max() + 2])
    fig.update_yaxes(title="True shooting %", tickformat=".0%", range=[SEASON["TS"].min() - 0.02, SEASON["TS"].max() + 0.02])
    if animate:
        btn_style = dict(bgcolor=THEME["nba_blue"], font=dict(color="#FFFFFF"), bordercolor=THEME["border"])
        fig.update_layout(
            updatemenus=[dict(type="buttons", showactive=False, x=0.0, y=-0.2, xanchor="left", yanchor="top", **btn_style, buttons=[
                dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=700, redraw=True), fromcurrent=True, transition=dict(duration=300))]),
                dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
            sliders=[dict(active=0, x=0.14, len=0.84, y=-0.12, yanchor="top", currentvalue=dict(prefix="Career year: ", font=dict(color=THEME["text"])),
                          font=dict(color=THEME["muted"]), bgcolor=THEME["panel_alt"], bordercolor=THEME["border"],
                          steps=[dict(method="animate", label=str(k), args=[[str(k)], dict(mode="immediate", frame=dict(duration=300, redraw=True), transition=dict(duration=200))])
                                 for k in range(1, maxk + 1)])])
    return fig


def fig_shot_profile(players):
    fig = make_subplots(rows=1, cols=2, subplot_titles=["3-point attempts per game", "3PA as a share of all field-goal attempts"], horizontal_spacing=0.08)
    fig.update_annotations(font=dict(color=THEME["text"], size=13))
    for k in (1, 2):
        for p in players:
            d = SEASON_BY_PLAYER[p].copy()
            col = "FG3A_PG" if k == 1 else "FG3A_RATE"
            g = with_gaps(d, "SEASON_START")
            fig.add_trace(go.Scatter(
                x=g["SEASON_START"], y=g[col], mode="lines+markers", name=p, legendgroup=p, showlegend=(k == 1), connectgaps=False,
                line=dict(color=PLAYER_COLORS[p], width=2.5, dash=DASHES[p]), marker=dict(symbol=SYMBOLS[p], size=7),
                customdata=g[["SEASON_YEAR"]].astype(object).values.tolist(),
                hovertemplate=f"<b>{p}</b> · %{{customdata[0]}}<br>" + ("3PA/G: %{y:.1f}" if k == 1 else "3PA share: %{y:.1%}") + "<extra></extra>"),
                row=1, col=k)
    style(fig, None, height=420, margin=dict(l=60, r=20, t=75, b=55))
    fig.update_layout(legend=dict(orientation="h", x=0.5, xanchor="center", y=1.12, yanchor="bottom"))
    fig.update_xaxes(title="Season start year (calendar time)")
    fig.update_yaxes(tickformat=".0%", row=1, col=2)
    return fig


AGING_METRICS = {"Points per game": "PPG", "True shooting %": "TS", "Assists per game": "AST_AVE",
                 "Rebounds per game": "REB_AVE", "Steals + blocks per game": "STOCKS"}


def fig_aging(label, players):
    col = AGING_METRICS[label]
    fig = go.Figure()
    for p in players:
        d = SEASON_BY_PLAYER[p].copy()
        peak = d.loc[d["QUALIFIED"], col].max()
        d["_IDX"] = d[col] / peak * 100
        g = with_gaps(d, "PLAYER_SEASON_NUM")
        low = g["LOW_GAMES"].eq(True).tolist()
        fig.add_trace(go.Scatter(
            x=g["PLAYER_SEASON_NUM"], y=g["_IDX"], mode="lines+markers", name=p, connectgaps=False,
            line=dict(color=PLAYER_COLORS[p], width=2.5, dash=DASHES[p]),
            marker=dict(symbol=SYMBOLS[p], size=8, color=[THEME["panel"] if lw else PLAYER_COLORS[p] for lw in low], line=dict(color=PLAYER_COLORS[p], width=2)),
            customdata=g[["SEASON_YEAR", col]].astype(object).values.tolist(),
            hovertemplate=f"<b>{p}</b> · %{{customdata[0]}}<br>Index: %{{y:.0f}} (peak = 100)<br>Raw: %{{customdata[1]:.3f}}<extra></extra>"))
    fig.add_hline(y=100, line=dict(color=THEME["muted"], width=1, dash="dot"))
    fig.add_hline(y=90, line=dict(color=THEME["muted"], width=1, dash="dash"),
                  annotation_text="90% of peak", annotation_font=dict(color=THEME["muted"], size=11), annotation_position="bottom right")
    style(fig, f"Aging curve: {label} indexed to each player's own peak (= 100)", height=450, margin=dict(l=65, r=30, t=75, b=55))
    fig.update_xaxes(title="Career year", dtick=1)
    fig.update_yaxes(title="% of own peak")
    return fig


def fig_durability(players, xmode):
    xcol, xtitle = xinfo(xmode)
    fig = make_subplots(rows=3, cols=1, vertical_spacing=0.1, row_heights=[0.38, 0.31, 0.31],
                        subplot_titles=["Regular-season games played", "Regular-season win %", "Playoff games as a share of all games played"])
    fig.update_annotations(font=dict(color=THEME["text"], size=13))
    for p in players:
        d = SEASON_BY_PLAYER[p]
        fig.add_trace(go.Bar(x=d[xcol], y=d["GAMES_PLAYED_REGULAR_SEASON"], name=p, legendgroup=p, marker_color=PLAYER_COLORS[p], opacity=0.9,
                             customdata=d[["SEASON_YEAR"]].values.tolist(),
                             hovertemplate=f"<b>{p}</b> · %{{customdata[0]}}<br>Games: %{{y}}<extra></extra>"), row=1, col=1)
        g = with_gaps(d, xcol)
        for r, col, f in ((2, "REGULAR_SEASON_WIN_PCT", "Win %"), (3, "PLAYOFF_SHARE", "Playoff share")):
            fig.add_trace(go.Scatter(
                x=g[xcol], y=g[col], mode="lines+markers", name=p, legendgroup=p, showlegend=False, connectgaps=False,
                line=dict(color=PLAYER_COLORS[p], width=2.5, dash=DASHES[p]), marker=dict(symbol=SYMBOLS[p], size=7),
                customdata=g[["SEASON_YEAR"]].astype(object).values.tolist(),
                hovertemplate=f"<b>{p}</b> · %{{customdata[0]}}<br>{f}: %{{y:.1%}}<extra></extra>"), row=r, col=1)
    fig.add_hline(y=82, line=dict(color=THEME["muted"], width=1, dash="dot"), row=1, col=1)
    style(fig, None, height=780, margin=dict(l=65, r=30, t=70, b=55))
    fig.update_layout(barmode="group", legend=dict(orientation="h", x=0.5, xanchor="center", y=1.06, yanchor="bottom"))
    fig.update_yaxes(tickformat=".0%", row=2, col=1)
    fig.update_yaxes(tickformat=".0%", row=3, col=1)
    fig.update_xaxes(title=xtitle, row=3, col=1)
    return fig


HIGHS = [("PTS_MAX", "Points"), ("REB_MAX", "Rebounds"), ("AST_MAX", "Assists"), ("STL_MAX", "Steals"), ("BLK_MAX", "Blocks"), ("TOV_MAX", "Turnovers")]


def fig_highs(players):
    fig = go.Figure()
    for p in players:
        v = [CAREER_X.loc[p, c] for c, _ in HIGHS]
        fig.add_trace(go.Bar(x=[l for _, l in HIGHS], y=v, name=p, marker_color=PLAYER_COLORS[p], text=[f"{x:.0f}" for x in v],
                             textposition="outside", hovertemplate=f"<b>{p}</b><br>%{{x}} career high: %{{y:.0f}}<extra></extra>"))
    style(fig, "Single-game career highs (regular season + playoffs)", height=420, margin=dict(l=60, r=30, t=75, b=55))
    fig.update_layout(barmode="group", bargap=0.25)
    fig.update_yaxes(title="Single-game maximum", range=[0, CAREER_X[[c for c, _ in HIGHS]].max().max() * 1.15])
    return fig


# ---- insights (every number is computed from the data) -----------------------------------------
def _q(p):
    return SEASON_BY_PLAYER[p].query("QUALIFIED").reset_index(drop=True)


def _nm(p):
    return f'<b style="color:{PLAYER_COLORS[p]}">{p}</b>'


def compute_insights(players):
    S = {p: _q(p) for p in players}
    cards = []

    # 1 · peak seasons
    lines = {}
    for p, q in S.items():
        i, j = q["PPG"].idxmax(), q["TS"].idxmax()
        same = "the same season" if q.loc[i, "SEASON_YEAR"] == q.loc[j, "SEASON_YEAR"] else "different seasons"
        lines[p] = (f"Scoring peak <b>{q.loc[i, 'SEASON_YEAR']}</b> ({q.loc[i, 'PPG']:.1f} PPG) · efficiency peak "
                    f"<b>{q.loc[j, 'SEASON_YEAR']}</b> ({q.loc[j, 'TS']:.1%} TS%) — {same}")
    top = max(S, key=lambda p: S[p]["PPG"].max())
    cards.append(("Peak seasons", f"{top} owns the highest single-season scoring average ({S[top]['PPG'].max():.1f} PPG)", lines))

    # 2 · biggest swings
    lines, best_jump = {}, {}
    for p, q in S.items():
        delta = q["PPG"].diff().where(q["SEASON_START"].diff() == 1)
        if delta.notna().any():
            i, j = delta.idxmax(), delta.idxmin()
            best_jump[p] = delta[i]
            lines[p] = f"Biggest jump <b>{q.loc[i, 'SEASON_YEAR']}</b> ({delta[i]:+.1f} PPG) · biggest drop <b>{q.loc[j, 'SEASON_YEAR']}</b> ({delta[j]:+.1f})"
        else:
            lines[p] = "Not enough consecutive seasons"
    head = (f"Largest one-year scoring jump: {max(best_jump, key=best_jump.get)} ({max(best_jump.values()):+.1f} PPG)" if best_jump else "n/a")
    cards.append(("Biggest year-over-year swings", head, lines))

    # 3 · prime length
    lines, n90 = {}, {}
    for p, q in S.items():
        n90[p] = int((q["PPG"] >= 0.9 * q["PPG"].max()).sum())
        lines[p] = f"<b>{n90[p]}</b> of {len(q)} qualifying seasons within 90% of his {q['PPG'].max():.1f} PPG peak"
    w = max(n90, key=n90.get)
    cards.append(("Prime length", f"{w} sustained a peak-level scoring prime longest ({n90[w]} seasons)", lines))

    # 4 · efficiency trajectory
    lines, gain = {}, {}
    for p, q in S.items():
        f3, l3 = q["TS"].head(3).mean(), q["TS"].tail(3).mean()
        gain[p] = (l3 - f3) * 100
        fall = (q["TS"].iloc[-1] - q["TS"].max()) * 100
        lines[p] = f"TS% first 3 seasons <b>{f3:.1%}</b> → last 3 <b>{l3:.1%}</b> ({gain[p]:+.1f} pts) · final season vs. peak: {fall:+.1f} pts"
    w = max(gain, key=gain.get)
    cards.append(("Efficiency trajectory", f"{w} improved shooting efficiency the most from early to late career ({gain[w]:+.1f} TS% pts)", lines))

    # 5 · shot profile
    lines, grow = {}, {}
    for p, q in S.items():
        f3, l3 = q["FG3A_PG"].head(3).mean(), q["FG3A_PG"].tail(3).mean()
        grow[p] = l3 - f3
        mult = f"{l3 / f3:.1f}×" if f3 > 0 else "n/a"
        lines[p] = f"3PA per game: first 3 seasons <b>{f3:.1f}</b> → last 3 <b>{l3:.1f}</b> ({mult})"
    w = max(grow, key=grow.get)
    cards.append(("Shot-profile shift", f"{w} added the most three-point volume ({grow[w]:+.1f} 3PA per game)", lines))

    # 6 · durability
    lines, avg = {}, {}
    for p in players:
        d = SEASON_BY_PLAYER[p]
        g = d["GAMES_PLAYED_REGULAR_SEASON"]
        avg[p] = g.mean()
        k = g.idxmin()
        lines[p] = f"<b>{avg[p]:.1f}</b> regular-season games per season · <b>{int((g < 60).sum())}</b> season(s) under 60 · lowest: {d.loc[k, 'SEASON_YEAR']} ({int(g[k])} games)"
    w = max(avg, key=avg.get)
    cards.append(("Durability", f"{w} played the most games per season ({avg[w]:.1f})", lines))

    # 7 · winning
    lines, wp = {}, {}
    for p in players:
        d = SEASON_BY_PLAYER[p]
        r = d[d["GAMES_PLAYED_REGULAR_SEASON"] >= MIN_GAMES_QUALIFIED]
        prime = r["QUALIFIED"] & (r["PPG"] >= 0.9 * S[p]["PPG"].max())
        wp[p] = r["REGULAR_SEASON_WIN_PCT"].max()
        b, wo = r["REGULAR_SEASON_WIN_PCT"].idxmax(), r["REGULAR_SEASON_WIN_PCT"].idxmin()
        non = r.loc[~prime, "REGULAR_SEASON_WIN_PCT"].mean()
        non_txt = f"{non:.1%}" if pd.notna(non) else "n/a"
        lines[p] = (f"Best <b>{r.loc[b, 'SEASON_YEAR']}</b> ({r.loc[b, 'REGULAR_SEASON_WIN_PCT']:.1%}) · worst <b>{r.loc[wo, 'SEASON_YEAR']}</b> "
                    f"({r.loc[wo, 'REGULAR_SEASON_WIN_PCT']:.1%}) · avg win % in prime seasons {r.loc[prime, 'REGULAR_SEASON_WIN_PCT'].mean():.1%} vs {non_txt} otherwise")
    w = max(wp, key=wp.get)
    cards.append(("Winning", f"{w} had the best single-season regular-season win rate ({wp[w]:.1%})", lines))

    # 8 · playmaking & rebounding
    lines, ast_gain = {}, {}
    for p, q in S.items():
        h = len(q) // 2
        if h < 2:
            lines[p] = "Not enough seasons"
            continue
        a1, a2 = q["AST_AVE"].iloc[:h].mean(), q["AST_AVE"].iloc[h:].mean()
        r1, r2 = q["REB_AVE"].iloc[:h].mean(), q["REB_AVE"].iloc[h:].mean()
        ast_gain[p] = a2 / a1 - 1
        ia, ir = q["AST_AVE"].idxmax(), q["REB_AVE"].idxmax()
        lines[p] = (f"Assists peaked <b>{q.loc[ia, 'SEASON_YEAR']}</b> ({q.loc[ia, 'AST_AVE']:.1f}), rebounds <b>{q.loc[ir, 'SEASON_YEAR']}</b> ({q.loc[ir, 'REB_AVE']:.1f}) · "
                    f"2nd-half vs 1st-half career: assists {a2 / a1 - 1:+.0%}, rebounds {r2 / r1 - 1:+.0%}")
    head = f"{max(ast_gain, key=ast_gain.get)} grew his playmaking most in the second half of his career ({max(ast_gain.values()):+.0%} assists)" if ast_gain else "n/a"
    cards.append(("Playmaking & rebounding evolution", head, lines))

    # 9 · title seasons vs the rest (external accolades table; small samples)
    lines, tp = {}, {}
    for p, q in S.items():
        is_t = q["SEASON_YEAR"].isin(season_list(p, "champion_seasons"))
        if is_t.any() and (~is_t).any():
            tp[p] = q.loc[is_t, "PPG"].mean()
            lines[p] = (f"<b>{int(is_t.sum())}</b> title seasons: {tp[p]:.1f} PPG · {q.loc[is_t, 'TS'].mean():.1%} TS% "
                        f"vs {q.loc[~is_t, 'PPG'].mean():.1f} PPG · {q.loc[~is_t, 'TS'].mean():.1%} TS% in his other seasons")
        else:
            lines[p] = "No title seasons among the qualifying seasons"
    head = f"{max(tp, key=tp.get)} scored the most in his title seasons ({max(tp.values()):.1f} PPG)" if tp else "n/a"
    cards.append(("Title seasons vs the rest · small samples", head, lines))
    return cards


def insights_html(players):
    out = []
    for title, head, lines in compute_insights(players):
        body = "".join(f'<div class="l">{_nm(p)} — {t}</div>' for p, t in lines.items())
        out.append(f'<div class="ins"><div class="t">{title}</div><div class="h">{head}</div>{body}</div>')
    return '<div class="ins-grid">' + "".join(out) + "</div>"


def big_picture_html():
    c = CAREER_X
    prime = {p: int((_q(p)["PPG"] >= 0.9 * _q(p)["PPG"].max()).sum()) for p in PLAYERS}
    tot, _, w, _ = goat_scores({k: 50 for k in CATS})
    bullets = [
        f"<b>Scoring:</b> {_nm(c['PTS_AVE'].idxmax())} has the highest career scoring average ({c['PTS_AVE'].max():.1f} PPG). For context only (not scored in the calculator), {_nm(c['PTS_TOTAL'].idxmax())} leads in total points ({c['PTS_TOTAL'].max():,.0f}).",
        f"<b>Efficiency:</b> {_nm(c['TS'].idxmax())} has the best career true-shooting percentage ({c['TS'].max():.1%}).",
        f"<b>All-around:</b> {_nm(c['AST_AVE'].idxmax())} leads in assists ({c['AST_AVE'].max():.1f}) and {_nm(c['REB_AVE'].idxmax())} in rebounds ({c['REB_AVE'].max():.1f}) per game; {_nm(c['STOCKS'].idxmax())} leads in steals + blocks ({c['STOCKS'].max():.2f}).",
        f"<b>Winning:</b> {_nm(c['REGULAR_SEASON_WIN_PCT'].idxmax())} owns the best regular-season win rate ({c['REGULAR_SEASON_WIN_PCT'].max():.1%}).",
        f"<b>Sustained peak:</b> {_nm(max(prime, key=prime.get))} has the longest peak-level scoring prime ({max(prime.values())} seasons within 90% of his best PPG); {_nm(c['GP_PER_SEASON'].idxmax())} averaged the most regular-season games per season ({c['GP_PER_SEASON'].max():.1f}).",
        f"<b>Hardware (external table):</b> {_nm(ACC['championships'].idxmax())} has the most titles ({ACC['championships'].max()}) and Finals MVPs ({ACC['finals_mvp'].max()}); {_nm(ACC['mvp'].idxmax())} the most league MVPs ({ACC['mvp'].max()}); {_nm(ACC['finals_win_pct'].idxmax())} the best Finals record ({int(ACC.loc[ACC['finals_win_pct'].idxmax(), 'championships'])}-{int(ACC.loc[ACC['finals_win_pct'].idxmax(), 'finals_appearances'] - ACC.loc[ACC['finals_win_pct'].idxmax(), 'championships'])}).",
        f"<b>The verdict depends on the question:</b> with equal weights the ranking is " + " › ".join(f"{_nm(p)} ({tot[p]:.1f})" for p in tot.index)
        + f"; across {MC_SIMULATIONS:,} random weightings {_nm(MC_FIRST.idxmax())} ranks first {MC_FIRST.max():.0f}% of the time.",
    ]
    return '<div class="bp"><ul style="margin:0;padding-left:20px">' + "".join(f"<li>{b}</li>" for b in bullets) + "</ul></div>"

In [ ]:
def build_trend_dashboard():
    boxes = {p: W.Checkbox(value=True, description=p, indent=False, layout=W.Layout(width="165px")) for p in PLAYERS}
    for p, b in boxes.items():
        b.add_class(f"pbox-{PLAYER_SLUG[p]}")
    xmode = W.ToggleButtons(options=["Career year", "Calendar season"], value="Career year", description="X-axis", style={"description_width": "50px"})
    smooth = W.Checkbox(value=False, description="3-season rolling average", indent=False, layout=W.Layout(width="230px"))
    awards_cb = W.Checkbox(value=True, description="Mark title ▲ and MVP ○ seasons", indent=False, layout=W.Layout(width="290px"))
    metric_dd = W.Dropdown(options=list(METRICS), value="Points per game", description="Metric", style={"description_width": "55px"}, layout=W.Layout(width="360px"))
    aging_dd = W.Dropdown(options=list(AGING_METRICS), value="Points per game", description="Aging metric", style={"description_width": "90px"}, layout=W.Layout(width="330px"))
    animate = W.Checkbox(value=False, description="Animate scatter by career year", indent=False, layout=W.Layout(width="260px"))
    names = ["trend", "multi", "heat", "cum", "scatter", "shot", "aging", "dur", "highs", "ins"]
    outs = {k: W.Output() for k in names}

    def players():
        return [p for p in PLAYERS if boxes[p].value]

    def draw(key, builder, cap=None):
        with outs[key]:
            clear_output(wait=True)
            ps = players()
            if not ps:
                display(empty_msg())
                return
            show(builder(ps))
            if cap:
                display(caption(cap(ps) if callable(cap) else cap))

    def r_trend(_=None):
        draw("trend", lambda ps: fig_trend(metric_dd.value, ps, xmode.value, smooth.value, awards_cb.value),
             "★ marks each player's best season for the selected metric (lowest for turnovers). Hollow markers = seasons with fewer than 30 games; "
             "gaps in the line = calendar seasons missing from the data. ▲ = NBA-title season, ○ = league-MVP season (external accolades table). "
             "Stats combine regular season and playoffs.")

    def r_multi(_=None):
        draw("multi", lambda ps: fig_small_multiples(ps, xmode.value, smooth.value),
             "Six core metrics on the same career clock — look for who peaks early, who sustains, and where each profile bends.")

    def r_heat(_=None):
        draw("heat", fig_season_heatmap,
             "Each row is z-scored against that player's own career, so red = a season well above his own average and blue = well below. "
             "For turnovers, red means <i>more</i> turnovers (not better). Blank cells are career years that player did not play.")

    def r_cum(_=None):
        draw("cum", fig_cumulative, cumulative_caption)

    def r_scatter(_=None):
        draw("scatter", lambda ps: fig_scatter(ps, animate.value),
             "Up and to the right is the best combination of volume and efficiency. Thin lines trace each career in order. Tick <i>Animate</i> to replay the careers year by year.")

    def r_shot(_=None):
        draw("shot", fig_shot_profile, "Always plotted on calendar time, because this chart is about the league-wide shift toward the three-point shot, not career age.")

    def r_aging(_=None):
        draw("aging", lambda ps: fig_aging(aging_dd.value, ps),
             "Every season is divided by that player's own best qualifying season. The longer a line stays above the 90% dashed line, the longer the player sustained his peak.")

    def r_dur(_=None):
        draw("dur", lambda ps: fig_durability(ps, xmode.value),
             "Dotted line = 82 games. Short seasons can reflect injury, retirement-year ramp-ups or league-shortened seasons (e.g. the 1998-99 lockout), so read them alongside the calendar gaps.")

    def r_highs(_=None):
        draw("highs", fig_highs, "Career-high single-game totals from the career summary file (regular season and playoffs combined).")

    def r_ins(_=None):
        with outs["ins"]:
            clear_output(wait=True)
            ps = players()
            display(empty_msg() if not ps else HTML(insights_html(ps)))

    def r_all(_=None):
        for f in (r_trend, r_multi, r_heat, r_cum, r_scatter, r_shot, r_aging, r_dur, r_highs, r_ins):
            f()

    for b in boxes.values():
        b.observe(r_all, names="value")
    xmode.observe(lambda _c: [f() for f in (r_trend, r_multi, r_dur)], names="value")
    smooth.observe(lambda _c: [f() for f in (r_trend, r_multi)], names="value")
    metric_dd.observe(r_trend, names="value")
    awards_cb.observe(r_trend, names="value")
    aging_dd.observe(r_aging, names="value")
    animate.observe(r_scatter, names="value")

    sec = lambda t: W.HTML(f'<div class="sec">{t}</div>')
    root = W.VBox([
        sec("Controls"),
        W.HBox([W.HTML("<b style='color:#8A96B0'>Players&nbsp;&nbsp;</b>")] + [boxes[p] for p in PLAYERS]),
        W.HBox([xmode, smooth, awards_cb]),
        sec("1 · Trend explorer"), metric_dd, outs["trend"],
        sec("2 · Core metrics side by side"), outs["multi"],
        sec("3 · Season-by-season heatmap"), outs["heat"],
        sec("4 · Cumulative career curves &amp; milestones"), outs["cum"],
        sec("5 · Efficiency vs volume"), animate, outs["scatter"],
        sec("6 · Shot-profile evolution"), outs["shot"],
        sec("7 · Aging &amp; decline curve"), aging_dd, outs["aging"],
        sec("8 · Durability, winning &amp; playoff presence"), outs["dur"],
        sec("9 · Single-game career highs"), outs["highs"],
        sec("Insights · computed from the data"), outs["ins"],
    ])
    root.add_class("goat-root")
    r_all()
    return root


TREND_UI = build_trend_dashboard()
display(TREND_UI)

### The big picture
Auto-generated from the data above (nothing here is typed in by hand).

In [ ]:
display(HTML('<div class="sec">The big picture</div>' + big_picture_html()))

## Methodology & limitations
* **Combined stats.** Per-game averages and shooting totals include regular season **and** playoffs; only games played and wins are split by season type, and wins/win % are regular season only.
* **Derived points.** The per-season file has no points column: `PTS = 2·FGM + 3PM + FTM`, `PPG = PTS / games (RS+PO)`. The career-level result reconciles with `PTS_AVE` (asserted in section 1). TS% = `PTS / (2·(FGA + 0.44·FTA))`, eFG% = `(FGM + 0.5·3PM) / FGA`.
* **Scoring model.** Each stat is scaled 0–100 against the best of the three players (inverted for lower-is-better stats). Stats are averaged into a category score using **your stat-level weights (1–10)**, and category scores are combined using **your category weights (1–10)**, so a stat's real influence = its share inside the category × the category's share of the total. Ratio scaling compresses near-ties (e.g. win %), so category gaps can look small even when they are real; the radar and heatmap show the category scores.
* **Accolades are external.** Championships, awards and honours are not in the CSVs; they come from the hand-compiled table in section 1b (as of 5 Nov 2026, cross-checked across NBA.com, ESPN and Wikipedia/Basketball-Reference-based lists). Verify or edit them there; a `data/external/player_accolades.csv` overrides the embedded values.
* **What the model leaves out on purpose.** Total points, total games and seasons played are not scored because they reward career length rather than greatness. All-Star / All-NBA / All-Defensive selections are scored per season played (`HONORS_AS_RATES = True`); switch it off to see raw totals, which favour the longest career.
* **Accolades carry their own biases.** Voting rules and eligibility change over time (e.g. LeBron's 2026 All-NBA streak ended under the league's 65-game minimum), All-Star/All-NBA counts depend on league size and era, and Finals records depend on teammates and opponents.
* **Still not in the data.** Advanced metrics (PER, WS, BPM) and player ages are not available and were not invented.
* **Small sample.** Three players means every ranking is relative, and the 4–6 title seasons behind the "title seasons" insight are tiny samples; the "peak", "prime" and "consistency" metrics are computed over seasons with at least 30 games.
* **Gaps & partial seasons.** `PLAYER_SEASON_NUM` ignores calendar gaps (e.g. Jordan's retirements). Lines break at gaps; low-game seasons (< 30 games) are drawn with hollow markers and excluded from peak/prime/consistency calculations. The most recent LeBron season may be incomplete depending on when the data was pulled.
* **A GOAT score is a model of *your* priorities**, not an objective truth — that is exactly what the sliders and the robustness check are for.

## Static snapshots (for GitHub previews)
GitHub does not render ipywidgets or Plotly JavaScript. This cell exports PNGs of the default-state charts to `notebooks/assets/` and shows them inline so the notebook (and any README that links to them) has visible charts on GitHub. Requires `pip install kaleido` and a local Chrome/Chromium; otherwise it is skipped harmlessly.

In [ ]:
def export_static():
    if not EXPORT_STATIC:
        print("EXPORT_STATIC is False — skipped.")
        return
    total, contrib, w, _ = goat_scores({k: 50 for k in CATS})
    figs = {
        "00_trophy_case": fig_trophies(),
        "01_goat_contributions": fig_contrib(total, contrib),
        "02_goat_radar": fig_radar(),
        "03_goat_category_winners": fig_heat(w),
        "04_goat_robustness": fig_mc(),
        "05_trend_ppg": fig_trend("Points per game", PLAYERS, "Career year", False),
        "06_core_metrics": fig_small_multiples(PLAYERS, "Career year", False),
        "07_season_heatmap": fig_season_heatmap(PLAYERS),
        "08_cumulative": fig_cumulative(PLAYERS),
        "09_efficiency_vs_volume": fig_scatter(PLAYERS, False),
        "10_shot_profile": fig_shot_profile(PLAYERS),
        "11_aging_curve": fig_aging("Points per game", PLAYERS),
        "12_durability": fig_durability(PLAYERS, "Career year"),
    }
    ASSETS_DIR.mkdir(exist_ok=True)
    try:
        for name, fig in figs.items():
            path = ASSETS_DIR / f"{name}.png"
            fig.write_image(str(path), width=1100, height=int(fig.layout.height or 450), scale=2)
            display(Image(filename=str(path)))
        print(f"Saved {len(figs)} PNG snapshots to {ASSETS_DIR.resolve()}")
    except Exception as exc:                       # kaleido / Chrome not available
        first = str(exc).strip().splitlines()[0] if str(exc).strip() else type(exc).__name__
        print("Static export skipped:", first)
        print("To enable: pip install kaleido  and install Chrome (or run: plotly_get_chrome).")


export_static()